In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Notebook used to perform the matching between items in the NDB list and items in foodDB using fuzzy matching

In [2]:
%cd drive/My Drive/FoodDB_NDB_matching/ ///

[Errno 2] No such file or directory: 'drive/My Drive/FoodDB_NDB_matching'
/content


In [3]:
import numpy as np
import pandas as pd
import json
import ast
import time
import copy
import os
from os import path

In [4]:
!pip install openpyxl
import openpyxl

In [ ]:
# file not available as foodDB item list not available for legal reasons - see data availability statement in accompanying paper.
fdb_df = pd.read_csv('products_with_categories.csv')
NDB_df = pd.read_csv('NDB_matching_data_FINAL.csv')

<ipython-input-5-9c641b1e22be>:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  fdb_df = pd.read_csv('products_with_categories.csv')


In [ ]:
NDB_df

,Unnamed: 0,Local description,Food composition record ID,NDNS Food group
0,0,7 Up free / light,8445,SOFT DRINKS LOW CALORIE CARBONATED
1,1,"Apple and blackcurrant squash / juice, with AD...",11581,SOFT DRINKS NOT LOW CALORIE CONCENTRATED
2,2,"Ackee, canned",1651,CANNED FRUIT IN JUICE
3,3,Aduki beans,8826,BEANS AND PULSES INCL READY MEAL & HOMEMADE DI...
4,4,"Adcal D3, calcium (600mg) with vitamin D (10ug...",10097,CALCIUM ONLY OR WITH VITAMIN D
...,...,...,...,...
3087,1582,"Chicken casserole, canned",1101,CHICKEN AND TURKEY DISHES
3088,1814,Hot dog/frankfurter with sauce in a bun,5308,SAUSAGES
3089,2036,McDonald's Caesar salad,8096,CHICKEN AND TURKEY DISHES
3090,2066,"Beef bolognese sauce, shop-bought",10576,BEEF VEAL AND DISHES


In [ ]:
!pip install fuzzywuzzy

In [ ]:
!pip install python-Levenshtein

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.9/172.9 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.9 MB/s eta 0:00:00


In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

In [ ]:
def save_mapping(mapping_dictionary, file_name):

  path = file_name

  for key, value in mapping_dictionary.items():
    if value is None:
        mapping_dictionary[key] = []

  df_to_save = pd.DataFrame.from_dict(mapping_dictionary, orient='index').reset_index()
  df_to_save.to_excel(path + '.xlsx', index=False, engine='openpyxl')

  with open(path+'.json', 'w') as json_file:
    json.dump(mapping_dictionary, json_file, indent=4)

  return


In [ ]:
def fuzzy_match(list_to_match, threshold, file_name):

  with open(file_name+ '.json', 'r') as json_file:
    mapping_dictionary = json.load(json_file)

  starting_index = list(mapping_dictionary.keys()).index("Waldorf salad")
  print(starting_index)

  items_to_check = list(mapping_dictionary.keys())[starting_index:]

  for idx, item in enumerate(items_to_check):

    possible_matches = process.extractBests(item, list_to_match, scorer=fuzz.ratio, limit=200)
    print(possible_matches)

    potential_matches = [match[0] for match in possible_matches if match[1] >= threshold]

    if potential_matches:

      existing_matches = mapping_dictionary.get(item, [])
      mapping_dictionary[item] = list(set(existing_matches) | set(potential_matches))
      print(item, mapping_dictionary[item])
      print('\n')

  save_mapping(mapping_dictionary=mapping_dictionary, file_name=file_name)

  return

In [ ]:
# file not available as foodDB item list not available for legal reasons - see data availability statement in accompanying paper.
path_file_name = 'Matching checks/Fuzzy_matching/NDB_to_FoodDB_FM_60'
fuzzy_match(list_to_match=fdb_df['product_list_name'].unique().tolist(), threshold=60, file_name=path_file_name)

In [ ]:
possible_matches = process.extract('Multivitamin with iron (14mg)', fdb_df['product_list_name'].unique().tolist() , scorer=fuzz.ratio)
potential_matches = [match[0] for match in possible_matches if match[1] >= 95]